## The State of Tax Justice: Impute Missing Data
- Author: Alison Schultz, based on Javier Garcia Bernado's work
- Created: 20 August 2023
- Last updated: 23 September 2024

**Description**
- This notebook is one out of three notebooks to estimate the tax losses caused by profit shifting by multinational enterprises (MNEs). The analysis uses the misalignment method based on the country-by-country reports (CbCR) published by the OECD.
- This notebook uses the extended CBCR dataset "data/intermediate/cbcr_imputation_sample.csv" created in the notebook "1_clean" and performs multiple impuration to replace missing values of country by country reporting data, mimicking a dataset of global MNE activity. It results in the notebook "data/final/cbcr_main.csv" that includes many datasets in which variables are imputed such that (i) variables from aggregated reporting done by some jurisdictions (e.g. per continent) are attributed to specific countries and (ii) global MNE activity from non-reporting countries is estimated. For details on the imputation process, see Appendix A1 in Garcia Bernardo/Janský (2024) that can be accessed here: https://www.sciencedirect.com/science/article/pii/S0305750X23003455, the working paper version is here: https://www.econstor.eu/bitstream/10419/286362/1/wp-2023-33.pdf 
- The notebook also generates descriptives to assess the imputation success.

**Outline**
1. Import data
2. Set up imputation process for domestic values
3. Set up imputation process for foreign values
4. Multiple imputation

**To dos before running this notebook**
- Run notebook 1_clean. Note the requirements and instructions given in that notebook.

### 0. Load packages

In [106]:
import pandas as pd
import numpy as np
from config import *
import tjn_tools
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from itertools import permutations, product
from sklearn.metrics import r2_score,make_scorer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LassoCV, Lasso, LinearRegression, BayesianRidge
from sklearn.experimental import enable_iterative_imputer, enable_hist_gradient_boosting
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.model_selection import cross_validate, train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, KFold
from sklearn.inspection import permutation_importance
from joblib import Parallel, delayed

### 1. Import data
- The imputation procedure relies on two datasets: First, cbcr_imputation_sample that includes the actual country by country reporting data from the OECD and all kinds of unilateral and bilateral variables that can be helpful to estimate missings in the actual OECD data. Second, cbcr_sample that only includes data that is actually reported for all subgroups of MNEs (i.e. aggregating both the ones that make profits and the ones that make losses).

In [107]:
cbcr_imputation_sample = pd.read_csv(f'{data_intermediate}/cbcr_imputation_sample.csv')
cbcr_sample = pd.read_csv(f'{data_intermediate}/cbcr_main_no_imputation_allsubgroupsonly.csv')

Some specific data cleaning that is only needed for the imputation process

In [108]:
# Correct mistake in Argentinian data
cbcr_imputation_sample.loc[
    (cbcr_imputation_sample['iso_parent'] == "ARG") & 
    (cbcr_imputation_sample['iso_partner'] == "S") & 
    (cbcr_imputation_sample['year'] == 2019), 
    'partner_iso'
] = 'S_O'

# "Rest of the world" is mainly defined as "WXD", but partly as "W_O".
cbcr_imputation_sample.loc[
    (cbcr_imputation_sample['iso_partner'] == "W_O"), 
    'partner_iso'
] = 'WXD'

Define target variables, i.e. the variables to be imputed. These are all variables that are or could be relevant to estimate misalignment

In [109]:
target_variables = ["profit_loss_before_income_tax_corrected",
    "ln_n_employees",
    "ln_unrelated_party_revenues",
    "ln_tangible_assets_except_cash",
    'ln_stated_capital',
    'ln_total_revenues',
    'ln_related_party_revenues',
    'ln_holding_or_managing_ip']

### 2. Set up imputation process for domestic values
- Domestic values are the values that a country reports about itself, e.g. the US activities of US based MNEs. We have to impute these values for all non-reporting countries

#### 2.1 Define imputation procedure for domestic values
- The following section imputes domestic variables, i.e. variables for which the reporting country (iso_parent) is the same as the partner country (iso_partner). These values are missing only for non-reporting countries, as all reporting countries specify the difference between domestic and foreign activity, even though they might not disaggregate foreign activity adequately.
- We estimate the number of domestic employees, revenue and tangible assets for all non-reporting countries. We do so by using a linear model based on the number of expected companies in each country (from Orbis), its GDP (from the Worldbank), population (from the Worldbank), the average ETRs (calculated in the notebook 1_clean) and the total consolidated banking claims on an immediate counterparty basis (from BIS). 



In [110]:
predictor_variables_domestic = ["ln_gdp_current_usd_partner", "ln_population_partner", "etr_foreign_corrected_partner", 
                             "cit_partner", "ln_n_companies_orbis", "ln_turnover_orbis", "ln_n_employees_orbis", 
                             "ln_wage_monthly_partner", "ln_bank_claims_total_partner",'cthi_2021_share_partner','cthi_2021_score_parent',
                             'wage_monthly_partner']

In [111]:
def predict_domestic_values(domestic_imputation_sample, variables_to_predict=target_variables, predictor_columns=predictor_variables_domestic):
    """
    This function imputes missing values for specified domestic variables using a linear regression model 
    based on a set of predictor variables. The function allows the user to specify both the variables to 
    predict (e.g., profits, employees, revenues) and the predictors (e.g., GDP, population, ETR, bank claims).

    Parameters:
    - domestic_imputation_sample: DataFrame containing the data to be used for imputation.
    - variables_to_predict: List of variables to predict (logged). Defaults to ['ln_profit_loss_before_income_tax_corrected', 
      'ln_n_employees', 'ln_unrelated_party_revenues', 'ln_tangible_assets_except_cash'].
    - predictor_columns: List of predictor variables to use in the regression model. Defaults to 
      ["ln_gdp_current_usd_partner", "ln_population_partner", "etr_foreign_corrected_partner", 
                             "cit_partner", "ln_n_companies_orbis", "ln_turnover_orbis", "ln_n_employees_orbis", 
                             "ln_wage_monthly_partner", "ln_bank_claims_total_partner", 'cthi_2021_share', 'cthi_2021_score']

    Returns:
    - DataFrame with original and imputed values for the specified variables.
    """

    # Initialize the imputer (using median strategy)
    imputer = SimpleImputer(strategy='median')

    for var in variables_to_predict:
        clf = LinearRegression()
        
        # Prepare the dataset
        df = domestic_imputation_sample[domestic_imputation_sample['year'].between(first_year, first_year + n_years - 1)].copy()
        
        # Ensure target variable is not missing
        df = df.dropna(subset=[var])
        
        # Filter rows where ln_n_companies_orbis > 0
        df = df[df["ln_n_companies_orbis"] > 0]
        
        # Extract predictors and target variable
        X = df[predictor_columns]
        y = df[var].values

        # Impute missing values in predictors for training data
        X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=predictor_columns, index=X.index)
        
        # Check if there are any samples left
        if X_imputed.shape[0] == 0:
            print(f"Skipping '{var}' because no valid samples are available for training (after imputing).")
            continue
        
        # Fit the model
        clf.fit(X_imputed, y)
        
        # Impute missing values in the predictors for the entire dataset
        X_full = domestic_imputation_sample[predictor_columns]
        X_full_imputed = pd.DataFrame(imputer.transform(X_full), columns=predictor_columns, index=X_full.index)

        # Predict for all rows and store in 'fitted_ln_variable'
        domestic_imputation_sample[f"fitted_{var}"] = clf.predict(X_full_imputed)
        
        # Set predicted value to 0 where ln_n_companies_orbis == 0
        # Set predicted values to 0 when any of the conditions are met
        domestic_imputation_sample.loc[
            (domestic_imputation_sample["ln_n_companies_orbis"] == 0) | 
            (domestic_imputation_sample["n_companies_orbis"] == 0) | 
            (domestic_imputation_sample["ln_n_companies_orbis"].isna()) | 
            (domestic_imputation_sample["n_companies_orbis"].isna()), 
            f"fitted_{var}"
        ] = 0
        
        # Convert predictions back to original scale if they were logged before and store in 'fitted_variable'
        if var.startswith("ln_"):
            non_log_var = var[3:]  # Remove 'ln_' prefix to get the non-logged variable name
            domestic_imputation_sample[f"fitted_{non_log_var}"] = np.expm1(domestic_imputation_sample[f"fitted_{var}"])  # Use np.expm1 instead of np.exp
        else:
            # If it's not logged, you can just assign it directly
            domestic_imputation_sample[f"fitted_{var}"] = domestic_imputation_sample[f"fitted_{var}"]
    # Set 'iso_partner' equal to 'iso_parent' (as imputation is done for domestic values)
    domestic_imputation_sample["iso_partner"] = domestic_imputation_sample["iso_parent"]
    
    return domestic_imputation_sample


#### 2.2 Evaluate domestic imputation success
Think about equalizing imputation method with foreign values and try to improve prediction for profits

In [112]:
def evaluate_domestic_prediction(domestic_imputation_sample, variables_to_predict=target_variables, predictor_columns=predictor_variables_domestic):
    """
    Evaluates regression models by calculating in-sample and out-of-sample R^2 and plotting fitted vs. actual values.
    
    Parameters:
    - domestic_imputation_sample: DataFrame containing the data.
    - variables_to_predict: List of target variables (logged). Defaults to the same as in predict_domestic_values().
    - predictor_columns: List of predictor variables. Defaults to the same as in predict_domestic_values().
    """

    # Initialize the imputer (using median strategy)
    imputer = SimpleImputer(strategy='median')

    # Dictionary to store R^2 scores
    r2_scores = {}

    for var in variables_to_predict:
        # Prepare the dataset
        df = domestic_imputation_sample[domestic_imputation_sample['year'].between(first_year, first_year + n_years - 1)].copy()

        # Ensure target variable is not missing
        df = df.dropna(subset=[var])

        # Filter rows where ln_n_companies_orbis > 0
        df = df[df["ln_n_companies_orbis"] > 0]

        # Extract predictors and target variable
        X = df[predictor_columns]
        y = df[var].values

        # Impute missing values in predictors for training data
        X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=predictor_columns, index=X.index)

        # Check if there are any samples left
        if X_imputed.shape[0] == 0:
            print(f"Skipping '{var}' because no valid samples are available for training (after imputing).")
            continue

        # Split data into training and test sets
        X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

        # Fit the model
        clf = LinearRegression()
        clf.fit(X_train, y_train)

        # Predict on training and test data
        y_train_pred = clf.predict(X_train)
        y_test_pred = clf.predict(X_test)

        # Calculate R^2 scores
        r2_train = r2_score(y_train, y_train_pred)
        r2_test = r2_score(y_test, y_test_pred)

        # Store R^2 scores
        r2_scores[var] = {'R2_train': r2_train, 'R2_test': r2_test}

        # Plotting the fitted vs. actual values for training data
        plt.figure(figsize=(8, 6))
        sns.scatterplot(x=y_train, y=y_train_pred, alpha=0.7)
        plt.title(f"Fitted vs. Actual Values for {var} (Training Data)")
        plt.xlabel("Actual Values")
        plt.ylabel("Fitted Values")
        plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
        plt.show()

        # Plotting the fitted vs. actual values for test data
        plt.figure(figsize=(8, 6))
        sns.scatterplot(x=y_test, y=y_test_pred, color='orange', alpha=0.7)
        plt.title(f"Fitted vs. Actual Values for {var} (Test Data)")
        plt.xlabel("Actual Values")
        plt.ylabel("Fitted Values")
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
        plt.show()

        print(f"Variable: {var}")
        print(f"In-sample R^2 (Training): {r2_train:.4f}")
        print(f"Out-of-sample R^2 (Test): {r2_test:.4f}")
        print("-" * 40)

    return r2_scores


Define domestic imputation sample to apply the evaluation

In [113]:
domestic_imputation_sample = cbcr_imputation_sample.loc[cbcr_imputation_sample["iso_parent"] == cbcr_imputation_sample["iso_partner"]]

Run evaluation

In [ ]:
evaluate_domestic_imputation = evaluate_domestic_prediction(domestic_imputation_sample)

### 3. Set up imputation process for foreign values
- These are the values for all combinations of reporting country (iso_parent) and partner country (iso_partner) for which:
    - the parent country is not reporting on an actual country by country but on an aggregate basis, or
    - the parent country is not reporting at all.

#### 3.1 Define metaparameters for imputation

Create foreign imputation sample to define metaparameters

In [115]:
foreign_imputation_sample = cbcr_imputation_sample.loc[cbcr_imputation_sample["iso_parent"] != cbcr_imputation_sample["iso_partner"]]
true_values = cbcr_sample[["iso_parent","iso_partner","year",'income_tax_paid_on_cash_basis','profit_loss_before_income_tax_corrected',
'n_employees',"unrelated_party_revenues","tangible_assets_except_cash",'stated_capital','total_revenues','related_party_revenues','holding_or_managing_ip']]

In [116]:
scorer = make_scorer(r2_score)

In [117]:
def find_best_model(est,clf,df):
        
    for var in ["profit_loss_before_income_tax_corrected",
    "ln_n_employees",
    "ln_unrelated_party_revenues",
    "ln_tangible_assets_except_cash",
    'ln_stated_capital',
    'ln_total_revenues',
    'ln_related_party_revenues',
    'ln_holding_or_managing_ip']:
        df = cbcr_sample.copy().dropna(subset=["profit_loss_before_income_tax_corrected",var])
        df[var].replace([np.inf, -np.inf], np.nan, inplace=True)
        print(var)
        y = df[var].values
        a = df.drop(columns=["income_tax_paid_on_cash_basis","profit_loss_before_income_tax_corrected","ln_n_employees","ln_unrelated_party_revenues",
            "ln_tangible_assets_except_cash",'ln_stated_capital','ln_total_revenues','ln_related_party_revenues','ln_holding_or_managing_ip']
            +[_ for _ in df.columns if "fitted" in _]+[_ for _ in df.columns if "month_wage" in _])
        columns = np.array(a.columns)[a.dtypes!=object]
        X = df[columns].values
        print(y.min(),y.mean(),y.max(),y.std())


        search = clf.fit(X,y)
        print(var, search.best_params_)
        try:
            plt.plot(search.param_grid["alpha"],search.cv_results_["mean_test_score"])
            plt.show()
        except:
            pass            

The following code block only has to be ran once to determine the meta parameters. If you want to run it, delete the "if 0:"

In [118]:
if 0: 
    distributions = dict(learning_rate=np.linspace(0.01,0.2,7),
                        l2_regularization=np.logspace(1,3,11),
                        min_samples_leaf=[20],)
    est = HistGradientBoostingRegressor(random_state=0)

    clf = GridSearchCV(est, distributions, n_jobs=-1, cv=5,verbose=5, scoring=scorer)

    find_best_model(est,clf,foreign_imputation_sample.dropna(subset=["profit_loss_before_income_tax_corrected"]))

In [119]:
# Set identified parameters
metaparameters  = {
    "profit_loss_before_income_tax_corrected": {'l2_regularization': 398.1071705534973, 'learning_rate': 0.1366666666666667, 'min_samples_leaf': 20},
    "ln_n_employees": {'l2_regularization': 39.810717055349734, 'learning_rate': 0.07333333333333333, 'min_samples_leaf': 20},
    "ln_unrelated_party_revenues": {'l2_regularization': 15.848931924611133, 'learning_rate': 0.105, 'min_samples_leaf': 20},
    "ln_tangible_assets_except_cash": {'l2_regularization': 39.810717055349734, 'learning_rate': 0.2, 'min_samples_leaf': 20},
    'ln_stated_capital': {'l2_regularization': 25.118864315095795, 'learning_rate': 0.07333333333333333, 'min_samples_leaf': 20},
    'ln_total_revenues': {'l2_regularization': 10.0, 'learning_rate': 0.105, 'min_samples_leaf': 20},
    'ln_related_party_revenues': {'l2_regularization': 39.810717055349734, 'learning_rate': 0.1366666666666667, 'min_samples_leaf': 20},
    'ln_holding_or_managing_ip': {'l2_regularization': 39.810717055349734, 'learning_rate': 0.1366666666666667, 'min_samples_leaf': 20}, #the last two rows are actually the same in the previously ran find_best_model() function
}

#### 3.2 Define imputation procedure for foreign values

Define predictor variables (target variables are the same as for the domestic imputation)

In [120]:
predictor_variables_foreign = ['etr_domestic_corrected_partner', 
                               'etr_foreign_corrected_partner', 
                               'etr_average_corrected_partner',
                               'cit_partner',
                               'population_partner',
                               'ln_wage_monthly_partner',
                               'ln_gdp_current_usd_partner',
                               'ln_gvt_health_expenditure_partner',
                               'ukt_partner',
                               'oecd_partner',
                               'nld_oct_partner',
                               'eu_partner',
                               'etr_domestic_corrected_parent', 
                               'etr_foreign_corrected_parent', 
                               'etr_average_corrected_parent',
                               'cit_parent',
                               'population_parent',
                               'ln_wage_monthly_parent',
                               'ln_gdp_current_usd_parent',
                               'ln_gvt_health_expenditure_parent',
                               'ukt_parent',
                               'oecd_parent',
                               'nld_oct_parent',
                               'eu_parent',
                               'ln_exports_tot',
                               'ln_imports_tot',
                               'ln_fdi_inward',
                               'ln_fdi_outward',
                               'ln_pi_inward',
                               'ln_pi_outward',
                               'ln_claims',
                               'ln_liabilities',
                               'ln_bank_claims_total_partner',
                               'ln_bank_claims_total_parent',
                               'distw_arithmetic',
                               'transition_legalchange',
                               'col45',
                               'col_dep_ever',
                               'comcol',
                               'comlang_ethno',
                               'comlang_off',
                               'comleg_posttrans',
                               'comleg_pretrans',
                               'comrelig',
                               'contig',
                               'sibling',
                               'fta_wto',
                               'r_heg',
                               'p_heg',
                               'r_entry_proc',
                               'p_entry_proc',
                               'r_entry_time',
                               'p_entry_time',
                               'r_entry_tp',
                               'p_entry_tp',
                               'r_entry_cost',
                               'p_entry_cost',
                               'r_gatt',
                               'p_gatt',
                               'both_eu',
                               'same_region',
                               'both_ukt',
                               'both_oecd',
                               'ln_n_companies_orbis',
                               'ln_n_employees_orbis',
                               'ln_turnover_orbis',
                               'wage_monthly_partner',
                               'wage_monthly_parent'
        ]

Define function to predict foreign values

In [121]:
def predict_foreign_values(foreign_imputation_sample, metaparameters=metaparameters, target_variables=target_variables, predictor_variables=predictor_variables_foreign):
    """
    Predicts foreign values using HistGradientBoostingRegressor with specified metaparameters.
    
    Parameters:
    - foreign_imputation_sample: DataFrame containing the data.
    - metaparameters: Dictionary containing hyperparameters for each target variable.
    - target_variables: List of target variables to predict (logged variables).
    - predictor_variables: List of predictor variables.
    
    Returns:
    - DataFrame with predicted values for the specified variables.
    """

    # Copy the dataset to avoid modifying the original
    data = foreign_imputation_sample.copy()

        # Loop over each target variable to predict
    for var in target_variables:
        print(f"Predicting values for: {var}")

        # Check if the target variable exists in the data
        if var not in data.columns:
            print(f"Target variable '{var}' not found in the data. Skipping...")
            continue

        # Create a training dataset: rows where the target variable is not missing
        train_df = data.dropna(subset=[var])

        # Ensure predictor variables exist in the data, excluding year and region_tjn
        available_predictors = [col for col in predictor_variables if col in data.columns]

        # Check if there are sufficient predictors
        if len(available_predictors) == 0:
            print(f"No available predictors for variable '{var}'. Skipping...")
            continue

        # Extract predictors (X) and target (y)
        X_train = train_df[available_predictors]
        y_train = train_df[var]

        # Retrieve metaparameters for the current variable
        params = metaparameters.get(var, {})
        if not params:
            print(f"No metaparameters found for variable '{var}'. Using default parameters.")

        # Initialize the model with metaparameters (if available)
        clf = HistGradientBoostingRegressor(
            max_iter=100,
            random_state=42,
            **params  # Unpack the metaparameters
        )

        # Fit the model on the training data
        clf.fit(X_train, y_train)

        # Predict on the entire dataset (including rows with missing target variable)
        X_full = data[available_predictors]

        # Predict the target variable (logged values) for all rows
        y_pred_logged = clf.predict(X_full)

        # Check if the variable is logged
        if var.startswith("ln_"):
            # Store the predicted logged values in a new column with prefix 'fitted_ln_'
            data[f"fitted_{var}"] = y_pred_logged  # Keeping the logged value

            # Remove 'ln_' prefix to get the non-logged variable name
            non_log_var = var[3:]

            # Exponentiate the predicted logged values to get predicted non-logged values
            y_pred_non_logged = np.expm1(y_pred_logged)  # Use np.expm1 for reverting np.log1p
            
            # Store the predicted non-logged values in a new column with prefix 'fitted_'
            data[f"fitted_{non_log_var}"] = y_pred_non_logged
        else:
            # If the variable is not logged, just assign the predicted values directly
            data[f"fitted_{var}"] = y_pred_logged  # Directly store the value without transformation

    # Return the dataset with the new 'fitted_' columns
    return data


#### 3.3 Check imputation success for foreign variables

In [122]:
def evaluate_foreign_predictions(data, metaparameters, target_variables=target_variables, predictor_variables=predictor_variables_foreign, test_size=0.2, random_state=42):
    """
    Evaluates the predictions by calculating in-sample and out-of-sample R² and plotting fitted vs. actual values.

    Parameters:
    - data: DataFrame containing the data with original variables.
    - metaparameters: Dictionary containing hyperparameters for each target variable.
    - target_variables: List of target variables to be predicted.
    - predictor_variables: List of predictor variables.
    - test_size: Proportion of the dataset to include in the test split.
    - random_state: Random seed for reproducibility.

    Returns:
    - Dictionary of R² scores for each variable.
    """

    # Copy the data to avoid modifying the original
    data = data.copy()

    r2_scores = {}

    for var in target_variables:
        print(f"Processing variable: {var}")

        # Check if the target variable exists in the data
        if var not in data.columns:
            print(f"Target variable '{var}' not found in the data. Skipping...")
            continue

        # Create a dataset: rows where the target variable is not missing
        df = data.dropna(subset=[var])

        # Ensure predictor variables exist in the data, excluding original year and region_tjn
        available_predictors = [col for col in predictor_variables if col in df.columns]

        # Check if there are sufficient predictors
        if len(available_predictors) == 0:
            print(f"No available predictors for variable '{var}'. Skipping...")
            continue

        # Extract predictors (X) and target (y)
        X = df[available_predictors]
        y = df[var]

        # Split the data into training and test sets
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state
        )

        # Retrieve metaparameters for the current variable
        params = metaparameters.get(var, {})
        if not params:
            print(f"No metaparameters found for variable '{var}'. Using default parameters.")

        # Initialize the model with metaparameters
        clf = HistGradientBoostingRegressor(
            max_iter=100,
            random_state=random_state,
            **params  # Unpack the metaparameters
        )

        # Fit the model on the training data
        clf.fit(X_train, y_train)

        # Predict on training data (in-sample)
        y_train_pred = clf.predict(X_train)

        # Predict on test data (out-of-sample)
        y_test_pred = clf.predict(X_test)

        # Calculate R² scores
        r2_train = r2_score(y_train, y_train_pred)
        r2_test = r2_score(y_test, y_test_pred)

        r2_scores[var] = {'R2_train': r2_train, 'R2_test': r2_test}

        # Plotting the fitted vs. actual values for training data
        plt.figure(figsize=(8, 6))
        sns.scatterplot(x=y_train, y=y_train_pred, alpha=0.7)
        plt.title(f"Fitted vs. Actual Values for {var} (Training Data)")
        plt.xlabel("Actual Values")
        plt.ylabel("Predicted Values")
        plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
        plt.show()

        # Plotting the fitted vs. actual values for test data
        plt.figure(figsize=(8, 6))
        sns.scatterplot(x=y_test, y=y_test_pred, color='orange', alpha=0.7)
        plt.title(f"Fitted vs. Actual Values for {var} (Test Data)")
        plt.xlabel("Actual Values")
        plt.ylabel("Predicted Values")
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
        plt.show()

        print(f"Variable: {var}")
        print(f"In-sample R² (Training): {r2_train:.4f}")
        print(f"Out-of-sample R² (Test): {r2_test:.4f}")
        print("-" * 40)

    return r2_scores


Run function to be able to evaluate its effectiveness

In [ ]:
filled_foreign_data = predict_foreign_values(foreign_imputation_sample,metaparameters)

Evaluate effectiveness of prediction

In [ ]:
r2_results = r2_results = evaluate_foreign_predictions(filled_foreign_data,metaparameters,target_variables,predictor_variables_foreign)

#### 3.4 Make sure that predicted values for individual countries add up to a known total

#### 3.4.1 Define scaling for continents

Define region country mapping

In [125]:
region_countries_mapping = {
    'A': [  # America
        'ATG', 'ARG', 'BHS', 'BRB', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL',
        'CRI', 'CUB', 'DMA', 'DOM', 'ECU', 'SLV', 'GRD', 'GTM', 'GUY', 'HTI',
        'HND', 'JAM', 'MEX', 'NIC', 'PAN', 'PRY', 'PER', 'KNA', 'LCA', 'VCT',
        'SUR', 'TTO', 'USA', 'URY', 'VEN'
        # Add or remove countries as needed
    ],
    'E': [  # Europe
        'ALB', 'AND', 'AUT', 'BLR', 'BEL', 'BIH', 'BGR', 'HRV', 'CYP', 'CZE',
        'DNK', 'EST', 'FIN', 'FRA', 'DEU', 'GRC', 'HUN', 'ISL', 'IRL', 'ITA',
        'LVA', 'LIE', 'LTU', 'LUX', 'MLT', 'MDA', 'MCO', 'MNE', 'NLD', 'MKD',
        'NOR', 'POL', 'PRT', 'ROU', 'RUS', 'SMR', 'SRB', 'SVK', 'SVN', 'ESP',
        'SWE', 'CHE', 'UKR', 'GBR', 'VAT'
        # Add or remove countries as needed
    ],
    'F': [  # Africa
        'DZA', 'AGO', 'BEN', 'BWA', 'BFA', 'BDI', 'CPV', 'CMR', 'CAF', 'TCD',
        'COM', 'COD', 'COG', 'CIV', 'DJI', 'EGY', 'GNQ', 'ERI', 'SWZ', 'ETH',
        'GAB', 'GMB', 'GHA', 'GIN', 'GNB', 'KEN', 'LSO', 'LBR', 'LBY', 'MDG',
        'MWI', 'MLI', 'MRT', 'MUS', 'MAR', 'MOZ', 'NAM', 'NER', 'NGA', 'RWA',
        'STP', 'SEN', 'SYC', 'SLE', 'SOM', 'ZAF', 'SSD', 'SDN', 'TZA', 'TGO',
        'TUN', 'UGA', 'ZMB', 'ZWE'
        # Add or remove countries as needed
    ],
    'S': [  # Asia
        'AFG', 'ARM', 'AZE', 'BHR', 'BGD', 'BTN', 'BRN', 'KHM', 'CHN', 'GEO',
        'IND', 'IDN', 'IRN', 'IRQ', 'ISR', 'JPN', 'JOR', 'KAZ', 'KWT', 'KGZ',
        'LAO', 'LBN', 'MYS', 'MDV', 'MNG', 'MMR', 'NPL', 'PRK', 'OMN', 'PAK',
        'PSE', 'PHL', 'QAT', 'SAU', 'SGP', 'KOR', 'LKA', 'SYR', 'TWN', 'TJK',
        'THA', 'TLS', 'TUR', 'TKM', 'ARE', 'UZB', 'VNM', 'YEM'
        # Add or remove countries as needed
    ],
}


In [126]:
# List of WXD jurisdictions (jurisdictions not tied to a specific continent)
wxd_jurisdictions = [
    'ABW',  # Aruba
    'AIA',  # Anguilla
    'BES',  # Bonaire, Sint Eustatius and Saba
    'BMU',  # Bermuda
    'COK',  # Cook Islands
    'CUW',  # Curaçao
    'CYM',  # Cayman Islands
    'GGY',  # Guernsey
    'GIB',  # Gibraltar
    'GLP',  # Guadeloupe
    'GUM',  # Guam
    'IMN',  # Isle of Man
    'JEY',  # Jersey
    'GIB',  # Gibraltar
    'HKG',  # Hong Kong
    'KNA',  # Saint Kitts and Nevis
    'MAC',  # Macao
    'MCO',  # Monaco
    'MSR',  # Montserrat
    'MTQ',  # Martinique
    'MYT',  # Mayotte
    'PRI',  # Puerto Rico
    'REU',  # Réunion
    'SHN',  # Saint Helena, Ascension and Tristan da Cunha
    'SXM',  # Sint Maarten (Dutch part)
    'TCA',  # Turks and Caicos Islands
    'VGB',  # British Virgin Islands
    'VIR',  # U.S. Virgin Islands
    'IOT',  # British Indian Ocean Territory

]

# Mapping of WXD jurisdictions to their respective continents
wxd_continent_mapping = {
    'A': [  # America
        'ABW',  # Aruba
        'AIA',  # Anguilla
        'BES',  # Bonaire, Sint Eustatius and Saba
        'BMU',  # Bermuda
        'CYM',  # Cayman Islands
        'CUW',  # Curaçao
        'GLP',  # Guadeloupe
        'GUM',  # Guam
        'KNA',  # Saint Kitts and Nevis
        'MSR',  # Montserrat
        'MTQ',  # Martinique
        'PAN',  # Panama
        'PRI',  # Puerto Rico
        'SXM',  # Sint Maarten (Dutch part)
        'TCA',  # Turks and Caicos Islands
        'VGB',  # British Virgin Islands
        'VIR',  # U.S. Virgin Islands
        # Add or remove jurisdictions as needed
    ],
    'E': [  # Europe
        'GGY',  # Guernsey
        'JEY',  # Jersey
        'IMN',  # Isle of Man
        'GIB',  # Gibraltar
        'MCO',  # Monaco
        # Add or remove jurisdictions as needed
    ],
    'F': [  # Africa
        'REU',  # Réunion
        'MYT',  # Mayotte
        'SHN',  # Saint Helena, Ascension and Tristan da Cunha
        # Add or remove jurisdictions as needed
    ],
    'S': [  # Asia
        'HKG',  # Hong Kong
        'MAC',  # Macao
        'IOT',  # British Indian Ocean Territory
        # Add or remove jurisdictions as needed
    ],
    # Additional mappings can be added for other continents if necessary
}

# Adjust the region mappings dynamically based on whether the iso_parent reports WXD
def update_region_mappings_for_wxd(region_countries_mapping, wxd_reported):
    updated_region_mapping = region_countries_mapping.copy()

    if not wxd_reported:
        # Include WXD jurisdictions in continent mappings
        for continent, countries in wxd_continent_mapping.items():
            updated_region_mapping[continent].extend(countries)
    
    return updated_region_mapping

Define variables to be scaled. These are always the same variables as target_variables but the non-logged version

In [127]:
scaling_variables = ["profit_loss_before_income_tax_corrected",
    "n_employees",
    "unrelated_party_revenues",
    "tangible_assets_except_cash",
    'stated_capital',
    'total_revenues',
    'related_party_revenues',
    'holding_or_managing_ip']

Specificy function for scaling

In [128]:
def scale_predicted_values_continents(filled_foreign_data, variables, region_countries_mapping):
    """
    Scales the predicted values of multiple variables for individual countries within aggregated regions,
    so that they sum up to the known values for the aggregated regions.

    Parameters:
    - filled_foreign_data: DataFrame containing the data, including the predicted values.
    - variables: List of variables to scale (e.g., ['n_employees', 'unrelated_party_revenues', ...]).
    - region_countries_mapping: A dictionary mapping aggregated regions to lists of country codes.

    Returns:
    - DataFrame with scaled predicted values for individual countries.
    """
    df = filled_foreign_data.copy()

    # Initialize a list to keep track of iso_parents where scaling was done
    iso_parents_scaled = []

    # For each aggregated region
    for region, countries in region_countries_mapping.items():
        print(f"Processing region: {region}")

        # For each iso_parent that has data for the aggregated region
        iso_parents = df[(df['iso_partner'] == region)  & (df['original_cbcr_data'] == 1)]['iso_parent'].unique()

        for iso_parent in iso_parents:
            # Check if the iso_parent reports "WXD"
            wxd_reported = 'WXD' in df[(df['iso_parent'] == iso_parent) & (df['original_cbcr_data'] == 1)]['iso_partner'].unique()

            # Adjust region mappings for this iso_parent
            updated_region_countries_mapping = update_region_mappings_for_wxd(region_countries_mapping, wxd_reported)

            # Record whether scaling was done for this iso_parent
            scaling_done = False

            # For each variable
            for variable in variables:
                predicted_var = f'fitted_{variable}'
                scaled_var = f'scaled_continent_{predicted_var}'

                if predicted_var not in df.columns:
                    print(f"Predicted variable '{predicted_var}' not found in DataFrame. Skipping variable '{variable}'.")
                    continue  # Skip to the next variable

                # Initialize the scaled variable column with NaN if not already initialized
                if scaled_var not in df.columns:
                    df[scaled_var] = np.nan

                # Get the known value for the aggregated region
                mask_region = (df['iso_partner'] == region) & (df['iso_parent'] == iso_parent) & (df['original_cbcr_data'] == 1)
                known_values = df.loc[mask_region, variable].values
                if len(known_values) == 0:
                    continue
                known_value = known_values[0]

                # Check if known_value is NaN
                if pd.isna(known_value):
                    continue  # Skip if known_value is NaN

                # Get the predicted values for individual countries within the region
                mask_countries = df['iso_partner'].isin(updated_region_countries_mapping[region]) & (df['iso_parent'] == iso_parent)
                predicted_values = df.loc[mask_countries, predicted_var]

                if predicted_values.empty:
                    continue

                sum_predicted = predicted_values.sum()

                # Check if sum_predicted is NaN or zero
                if pd.isna(sum_predicted) or sum_predicted == 0:
                    scaling_factor = np.nan
                else:
                    scaling_factor = known_value / sum_predicted

                # Check if scaling_factor is NaN
                if pd.isna(scaling_factor):
                    continue  # Skip scaling if scaling_factor is NaN

                # Scale the predicted values
                df.loc[mask_countries, scaled_var] = predicted_values * scaling_factor

                scaling_done = True  # Mark that scaling was done for this iso_parent

                # Only print if scaling_factor is not NaN
                print(f"  Scaled '{variable}' for iso_parent '{iso_parent}' in region '{region}' with scaling factor {scaling_factor:.4f}")

            # Record the iso_parent if scaling was done for any variable
            if scaling_done:
                iso_parents_scaled.append(iso_parent)

    # Optionally, print the iso_parents where scaling was done
    unique_iso_parents_scaled = list(set(iso_parents_scaled))
    print("\nScaling was performed for the following iso_parents:")
    for ip in unique_iso_parents_scaled:
        print(ip)

    return df

#### 3.4.2 Verify scaling according to continents

In [129]:
def verify_scaling_continents(df, variables, region_countries_mapping):
    """
    Verifies the scaling of predicted values for individual countries within aggregated regions.

    Parameters:
    - df: DataFrame containing the data with both original and scaled variables.
    - variables: List of variables that were scaled.
    - region_countries_mapping: A dictionary mapping aggregated regions to lists of country codes.

    Returns:
    - None
    """
    
    for variable in variables:
        scaled_var = f'scaled_continent_fitted_{variable}'
        print(f"\nVerifying scaling for variable: {variable}")

        # For each aggregated region
        for region, countries in region_countries_mapping.items():
            # Get iso_parents that report on this region
            iso_parents = df[(df['iso_partner'] == region) & (df['original_cbcr_data'] == 1)]['iso_parent'].unique()
            
            for iso_parent in iso_parents:
                # Get the known value for this aggregated region
                known_values = df.loc[
                    (df['iso_partner'] == region) & (df['iso_parent'] == iso_parent),
                    variable
                ].values
                
                if len(known_values) == 0:
                    print(f"No known values for iso_parent '{iso_parent}' in region '{region}'. Skipping...")
                    continue

                known_value = known_values[0]
                
                # Check if known_value is NaN
                if pd.isna(known_value):
                    print(f"Known value for iso_parent '{iso_parent}' is NaN. Skipping...")
                    continue  # Skip if known_value is NaN

                # Calculate the sum of scaled values for this iso_parent
                mask = df['iso_parent'] == iso_parent
                sum_scaled = df.loc[mask & df['iso_partner'].isin(countries), scaled_var].sum()

                # Check if sum_scaled is NaN
                if pd.isna(sum_scaled):
                    print(f"Sum of scaled values for iso_parent '{iso_parent}' is NaN. Skipping...")
                    continue  # Skip if sum_scaled is NaN
                
                # Compare the sum of scaled values to the known value
                if sum_scaled != known_value:
                    print(f"Discrepancy found for iso_parent '{iso_parent}' in region '{region}':")
                    print(f"  Known Value: {known_value}, Sum of Scaled Values: {sum_scaled}")
                else:
                    print(f"Scaling verification passed for iso_parent '{iso_parent}' in region '{region}'.")

In [ ]:
scaled_data_continents = scale_predicted_values_continents(
    filled_foreign_data=filled_foreign_data[filled_foreign_data['year'] == 2016],
    variables=scaling_variables,
    region_countries_mapping=region_countries_mapping
)

In [ ]:
verify_scaling_continents(scaled_data_continents, scaling_variables, region_countries_mapping)

#### 3.4.3 Define scaling for "other" countries (i.e. countries in a continent where other values are given on a country by country basis but some countries are subsumed under "Other")

In [132]:
region_other_mapping = {
    'A': 'A_O',  # "Other America"
    'E': 'E_O',  # "Other Europe"
    'F': 'F_O',  # "Other Africa"
    'S': 'S_O',  # "Other Asia"
}

In [133]:
def scale_other_predicted_values(filled_foreign_data, variables, region_countries_mapping, region_other_mapping):
    """
    Scales the predicted values of variables for "Other" groups in regions for each iso_parent.

    Parameters:
    - filled_foreign_data: DataFrame containing the data, including predicted values.
    - variables: List of variables to scale.
    - region_countries_mapping: Dictionary mapping regions to lists of country codes.
    - region_other_mapping: Dictionary mapping regions to their "Other" codes.

    Returns:
    - DataFrame with scaled predicted values for "Other" countries.
    """
    df = filled_foreign_data.copy()

    # Initialize scaled variable columns
    for variable in variables:
        scaled_var = f'scaled_other_fitted_{variable}'
        if scaled_var not in df.columns:
            df[scaled_var] = np.nan

    # List to keep track of iso_parents where scaling was done
    iso_parents_scaled = []

    # For each region
    for region_code, countries in region_countries_mapping.items():
        other_code = region_other_mapping.get(region_code)
        if not other_code:
            continue  # Skip if no "Other" code defined for the region

        print(f"Processing region: {region_code} (Other code: {other_code})")

        # Find iso_parents that have "Other" data for this region
        iso_parents = df[(df['iso_partner'] == other_code) & (df['original_cbcr_data'] == 1)]['iso_parent'].unique()

        for iso_parent in iso_parents:
            # Check if the iso_parent reports "WXD"
            wxd_reported = 'WXD' in df[(df['iso_parent'] == iso_parent) & (df['original_cbcr_data'] == 1)]['iso_partner'].unique()

            # Adjust region mappings for this iso_parent
            updated_region_countries_mapping = update_region_mappings_for_wxd(region_countries_mapping, wxd_reported)

            # For each variable
            scaling_done = False
            for variable in variables:
                predicted_var = f'fitted_{variable}'
                scaled_var = f'scaled_other_fitted_{variable}'

                # Ensure the predicted variable exists
                if predicted_var not in df.columns:
                    print(f"Predicted variable '{predicted_var}' not found. Skipping variable '{variable}'.")
                    continue

                # Get the known "Other" value
                mask_other = (df['iso_partner'] == other_code) & (df['iso_parent'] == iso_parent)  & (df['original_cbcr_data'] == 1)
                known_values = df.loc[mask_other, variable].values
                if len(known_values) == 0:
                    continue
                known_other_value = known_values[0]

                # Check if known_other_value is NaN
                if pd.isna(known_other_value):
                    continue  # Skip if known_other_value is NaN

                # Identify countries in the region where the iso_parent does NOT have individual data
                mask_countries_in_other = (df['iso_partner'].isin(updated_region_countries_mapping[region_code])) & (df['iso_parent'] == iso_parent) & (df['original_cbcr_data'] != 1)
                
                # Get predicted values for these countries
                predicted_values = df.loc[mask_countries_in_other, predicted_var]
                
                if predicted_values.empty:
                    continue
                
                sum_predicted = predicted_values.sum()
                
                # Check if sum_predicted is NaN or zero
                if pd.isna(sum_predicted) or sum_predicted == 0:
                    scaling_factor = np.nan
                else:
                    scaling_factor = known_other_value / sum_predicted
                
                # Check if scaling_factor is NaN
                if pd.isna(scaling_factor):
                    continue  # Skip scaling if scaling_factor is NaN

                # Scale the predicted values
                df.loc[mask_countries_in_other, scaled_var] = predicted_values * scaling_factor
                
                scaling_done = True
                
                # Only print if scaling_factor is not NaN
                print(f"  Scaled '{variable}' for iso_parent '{iso_parent}' in 'Other' of region '{region_code}' with scaling factor {scaling_factor:.4f}")
            
            if scaling_done:
                iso_parents_scaled.append(iso_parent)
    
    # Optionally, print the iso_parents where scaling was done
    unique_iso_parents_scaled = list(set(iso_parents_scaled))
    print("\nScaling for 'Other' was performed for the following iso_parents:")
    for ip in unique_iso_parents_scaled:
        print(ip)
    
    return df


In [134]:
def verify_other_scaling(df, variables, region_countries_mapping, region_other_mapping):
    """
    Verifies the scaling of predicted values for 'Other' groups in regions.

    Parameters:
    - df: DataFrame containing the data with both original and scaled variables.
    - variables: List of variables that were scaled.
    - region_countries_mapping: Dictionary mapping regions to lists of country codes.
    - region_other_mapping: Dictionary mapping regions to their "Other" codes.

    Returns:
    - None
    """
    
    for variable in variables:
        scaled_var = f'scaled_other_fitted_{variable}'
        print(f"\nVerifying scaling for 'Other' variable: {variable}")

        for region_code, countries in region_countries_mapping.items():
            other_code = region_other_mapping.get(region_code)
            if not other_code:
                continue  # Skip if no "Other" code defined for the region
            
            # Get iso_parents that report on this region's "Other" code
            iso_parents = df[df['iso_partner'] == other_code]['iso_parent'].unique()
            
            for iso_parent in iso_parents:
                # Known "Other" value
                known_values = df.loc[
                    (df['iso_partner'] == other_code) & (df['iso_parent'] == iso_parent) & (df['original_cbcr_data'] == 1),
                    variable
                ].values
                
                if len(known_values) == 0:
                    print(f"No known values for iso_parent '{iso_parent}' in 'Other' for region '{region_code}'. Skipping...")
                    continue
                
                known_other_value = known_values[0]
                
                # Check if known_other_value is NaN
                if pd.isna(known_other_value):
                    print(f"Known 'Other' value for iso_parent '{iso_parent}' is NaN. Skipping...")
                    continue  # Skip if known_other_value is NaN

                # Sum of scaled predicted values
                mask_region_countries = (df['iso_partner'].isin(countries)) & (df['iso_parent'] == iso_parent) & (df['original_cbcr_data'] != 1)
                sum_scaled = df.loc[mask_region_countries, scaled_var].sum()
                
                # Check if sum_scaled is NaN
                if pd.isna(sum_scaled):
                    print(f"Sum of scaled values for iso_parent '{iso_parent}' in 'Other' region '{region_code}' is NaN. Skipping...")
                    continue  # Skip if sum_scaled is NaN
                
                # Compare the sum of scaled values to the known value
                if sum_scaled != known_other_value:
                    print(f"Discrepancy found for iso_parent '{iso_parent}' in 'Other' region '{region_code}':")
                    print(f"  Known 'Other' Value: {known_other_value}, Sum of Scaled Predictions: {sum_scaled}")
                else:
                    print(f"Scaling verification passed for iso_parent '{iso_parent}' in 'Other' region '{region_code}'.")

In [ ]:
scaled_data_other = scale_other_predicted_values(
    filled_foreign_data=filled_foreign_data[filled_foreign_data['year'] == 2016],
    variables=scaling_variables,
    region_countries_mapping=region_countries_mapping,
    region_other_mapping=region_other_mapping
)

In [ ]:
verify_other_scaling(scaled_data_other,
                     variables=scaling_variables,
                     region_countries_mapping=region_countries_mapping,
                     region_other_mapping=region_other_mapping)

#### 3.4.4 Define scaling for the countries that are aggregated under "Rest of the world" ("WXD")

Create the scale_wxd_values Function

In [137]:
def scale_wxd_values(foreign_data, variables, wxd_jurisdictions, region_countries_mapping):
    """
    Scales the predicted values for the 'WXD' group based on the reporting practices of each iso_parent.
    
    Parameters:
    - foreign_data: DataFrame containing foreign observations with predicted values.
    - variables: List of variables to scale.
    - wxd_jurisdictions: List of country codes for 'WXD' jurisdictions.
    - region_countries_mapping: Mapping of regions to their respective countries.
    
    Returns:
    - DataFrame with scaled predicted values for 'WXD' jurisdictions.
    """
    df = foreign_data.copy()
    
    # List to keep track of iso_parents where scaling was done
    iso_parents_scaled = []
    
    # Get the list of iso_parents that report on 'WXD'
    iso_parents_wxd = df[(df['iso_partner'] == 'WXD') & (df['original_cbcr_data'] == 1)]['iso_parent'].unique()
    
    for iso_parent in df['iso_parent'].unique():
        df_parent = df[df['iso_parent'] == iso_parent]
        reports_wxd = iso_parent in iso_parents_wxd
        
        # Update the region mapping correctly
        updated_region_countries_mapping = update_region_mappings_for_wxd(region_countries_mapping, reports_wxd)
        
        if reports_wxd:
            # Get known 'WXD' values for this iso_parent
            known_wxd_data = df_parent[df_parent['iso_partner'] == 'WXD']
            
            # Distinguish between partner countries in original data and countries not reported on
            iso_partners_reported = df_parent[(df_parent['iso_partner'] != 'WXD') &
                                              (df_parent['iso_partner'] != df_parent['iso_parent']) &
                                              (df_parent['original_cbcr_data'] == 1)]['iso_partner'].unique()
            iso_partners_not_reported = df_parent[(df_parent['iso_partner'] != 'WXD') &
                                                  (df_parent['iso_partner'] != df_parent['iso_parent']) &
                                                  (df_parent['original_cbcr_data'] != 1)]['iso_partner'].unique() 
            
            # Case 1: WXD and regions or other regions are reported in original data
            # --> Only scale countries as WXD that do not have cbcr_original_data == 1 and that are part of the wxd_jurisdictions list
            #     and not reported in the original data of the given iso_parent
            if any(partner in other_country_groups for partner in iso_partners_reported) or \
               any(partner in aggregated_country_groups for partner in iso_partners_reported):
                countries_to_scale = [
                    country for country in wxd_jurisdictions
                    if country in iso_partners_not_reported
                ]
            else:                                    
                # Case 2: WXD is reported in original data but no regions or other regions
                # ---> WXD encompasses all jurisdictions not reported individually
                countries_to_scale = iso_partners_not_reported
        else:
            # Case 3: No 'WXD' reported
            # --> no scaling with this function (but parts of the WXD countries might be scaled via their continent or "other" category)
            continue 
        
        # Scale the predicted values for the determined countries
        for variable in variables:
            predicted_var = f'fitted_{variable}'
            scaled_var = f'scaled_wxd_fitted_{variable}'
            if scaled_var not in df.columns:
                df[scaled_var] = np.nan
            
            # Get known 'WXD' value
            known_wxd_value = known_wxd_data[variable].values[0] if not known_wxd_data.empty else np.nan
            
            # Get predicted values for countries to scale
            mask_countries_to_scale = (df['iso_parent'] == iso_parent) & (df['iso_partner'].isin(countries_to_scale))
            predicted_values = df.loc[mask_countries_to_scale, predicted_var]
            
            if predicted_values.empty:
                continue
            
            sum_predicted = predicted_values.sum()
            
            # Check for division by zero
            if pd.isna(known_wxd_value) or sum_predicted == 0 or pd.isna(sum_predicted):
                scaling_factor = np.nan
            else:
                scaling_factor = known_wxd_value / sum_predicted
            
            # Apply scaling if scaling_factor is valid
            if pd.isna(scaling_factor):
                continue
            
            df.loc[mask_countries_to_scale, scaled_var] = predicted_values * scaling_factor
            iso_parents_scaled.append(iso_parent)
    
    # Optionally, print the iso_parents where scaling was done
    unique_iso_parents_scaled = list(set(iso_parents_scaled))
    print(f"\nScaling for 'WXD' was performed for {len(unique_iso_parents_scaled)} iso_parents.")
    
    return df


In [138]:
def verify_wxd_scaling(df, variables, other_country_groups, aggregated_country_groups):
    """
    Verifies the scaling of WXD values to ensure that the summed scaled values match the known WXD values.

    Parameters:
    - df: DataFrame containing the data with both original and scaled variables.
    - variables: List of variables that were scaled.
    - other_country_groups: List or set of other country groups for the checks.
    - aggregated_country_groups: List or set of aggregated country groups for the checks.

    Returns:
    - None
    """
    
    # Get the list of iso_parents that report on 'WXD'
    iso_parents_wxd = df[(df['iso_partner'] == 'WXD') & (df['original_cbcr_data'] == 1)]['iso_parent'].unique()
    
    for iso_parent in iso_parents_wxd:
        # Get the known WXD values for this iso_parent
        known_wxd_data = df[(df['iso_parent'] == iso_parent) & (df['iso_partner'] == 'WXD')]
        
        if known_wxd_data.empty:
            print(f"No known WXD data for iso_parent '{iso_parent}'. Skipping...")
            continue
        
        # Distinguish reported and not reported iso_partners
        iso_partners_reported = df[(df['iso_parent'] == iso_parent) & (df['iso_partner'] != 'WXD') & (df['original_cbcr_data'] == 1)]['iso_partner'].unique()
        iso_partners_not_reported = df[(df['iso_parent'] == iso_parent) & (df['iso_partner'] != 'WXD') & (df['original_cbcr_data'] != 1)]['iso_partner'].unique() 

        # Define countries_to_scale based on the logic used in your scaling function
        if any(partner in other_country_groups for partner in iso_partners_reported) or \
           any(partner in aggregated_country_groups for partner in iso_partners_reported):
            countries_to_scale = [
                country for country in wxd_jurisdictions
                if country not in iso_partners_reported
            ]
        else:                                    
            countries_to_scale = iso_partners_not_reported

        for var in variables:
            # Sum the known WXD value for this variable
            known_wxd_value = known_wxd_data[var].sum()  # Assuming multiple known values
            
            # Get the scaled values for this iso_parent
            scaled_var = f'scaled_wxd_fitted_{var}'
            mask = (df['iso_parent'] == iso_parent) & (df['iso_partner'].isin(countries_to_scale))
            scaled_values = df.loc[mask, scaled_var]
            
            # Calculate the sum of scaled values
            sum_scaled = scaled_values.sum()
            
            # Check if the sum of scaled values matches the known WXD value
            if pd.isna(known_wxd_value) or pd.isna(sum_scaled):
                print(f"Skipping verification for iso_parent '{iso_parent}' and variable '{var}' due to NaN values.")
                continue  # Skip if known or scaled values are NaN
            
            # Round the sums to one decimal place for comparison
            known_wxd_value_rounded = round(known_wxd_value, 1)
            sum_scaled_rounded = round(sum_scaled, 1)

            # Compare the rounded sum of scaled values to the rounded known value
            if sum_scaled_rounded != known_wxd_value_rounded:
                print(f"Discrepancy found for iso_parent '{iso_parent}' and variable '{var}':")
                print(f"  Known WXD value (rounded): {known_wxd_value_rounded} (summed known values: {known_wxd_data[var].values})")
                print(f"  Sum of scaled values (rounded): {sum_scaled_rounded}")
            else:
                print(f"Scaling verification passed for iso_parent '{iso_parent}' and variable '{var}'.")


In [ ]:
scaled_data_wxd = scale_wxd_values(
    foreign_data=filled_foreign_data[filled_foreign_data['year'] == 2016],
    variables=scaling_variables,
    wxd_jurisdictions=wxd_jurisdictions,
    region_countries_mapping=region_countries_mapping
)


In [ ]:
verify_wxd_scaling(
    scaled_data_wxd,
    variables=scaling_variables,
    other_country_groups=other_country_groups,
    aggregated_country_groups = aggregated_country_groups
    )

### 4. Multiple imputation
- Now, all the previously defined steps are combined to:
    1. Impute missing domestic values
    2. Impute missing foreign values
    3. Make sure that imputed values add up to known totals:
        - For aggregated continent data
        - for aggregated "other" continent data
        - for aggregated "other" / "rest of the world" countries
- This imputation is ran n times to generate a range of potential dataset with slightly different imputed values. Later (in notebook 3_estimate_profit_shifting), we will calculate misalignment for each of these datasets and take the median value.


In [ ]:
# Replace missing ETRs (only two in sample) with CIT and log affected iso_partner
etr_missing_before = cbcr_sample[cbcr_sample['etr_average_corrected'].isna()]['iso_partner'].unique()
cbcr_sample['etr_average_corrected'] = cbcr_sample['etr_average_corrected'].fillna(cbcr_sample['cit'])
etr_missing_after = cbcr_sample[cbcr_sample['etr_average_corrected'].isna()]['iso_partner'].unique()

# Output the partners that had ETRs filled with CIT values
etr_filled = set(etr_missing_before) - set(etr_missing_after)
if etr_filled:
    print(f"ETR was filled using CIT for iso_partner(s): {', '.join(etr_filled)}")
else:
    print("No missing ETRs were filled.")

# Replace missing CIT rates with ETR and log affected iso_partner
cit_missing_before = cbcr_sample[cbcr_sample['cit'].isna()]['iso_partner'].unique()
cbcr_sample['cit'] = cbcr_sample['cit'].fillna(cbcr_sample['etr_average_corrected'])
cit_missing_after = cbcr_sample[cbcr_sample['cit'].isna()]['iso_partner'].unique()

# Output the partners that had CIT rates filled with ETR values
cit_filled = set(cit_missing_before) - set(cit_missing_after)
if cit_filled:
    print(f"CIT was filled using ETR for iso_partner(s): {', '.join(cit_filled)}")
else:
    print("No missing CITs were filled.")

#### 4.1 Illustrate imputation process for one sample

In [ ]:
# Initialize a list to collect results for each year
final_results = []

# Iterate over the range of years
for year in range(first_year, first_year + n_years):
    print(f"Processing year: {year}")

    # Step 1: Split the DataFrame for the current year
    cbcr_yearly_data = cbcr_imputation_sample[cbcr_imputation_sample['year'] == year]

    domestic_mask = cbcr_yearly_data['iso_parent'] == cbcr_yearly_data['iso_partner']
    foreign_mask = cbcr_yearly_data['iso_parent'] != cbcr_yearly_data['iso_partner']

    cbcr_domestic = cbcr_yearly_data[domestic_mask].copy()
    cbcr_foreign = cbcr_yearly_data[foreign_mask].copy()

    # Step 2: Drop rows which we do not need to impute
    #partner_year_groups = cbcr_foreign.groupby(['iso_partner', 'year'])
    #for (partner, year), group in partner_year_groups:
    #    if (group['original_cbcr_data'] == 1).any() and not any(group['iso_partner'].isin(non_countries)):
    #        cbcr_foreign = cbcr_foreign[~((cbcr_foreign['iso_partner'] == partner) & 
    #                                      (cbcr_foreign['year'] == year) & 
    #                                     (cbcr_foreign['original_cbcr_data'] == 0))]

    # Step 3: Impute Domestic Values
    cbcr_domestic_imputed = predict_domestic_values(cbcr_domestic)

    # Step 4: Impute Foreign Values
    cbcr_foreign_imputed = predict_foreign_values(cbcr_foreign)

    # Step 5: Scaling for different groups
    cbcr_foreign_scaled1 = scale_predicted_values_continents(
        filled_foreign_data=cbcr_foreign_imputed,
        variables=scaling_variables,
        region_countries_mapping=region_countries_mapping
    )

    cbcr_foreign_scaled2 = scale_other_predicted_values(
        filled_foreign_data=cbcr_foreign_scaled1,
        variables=scaling_variables,
        region_countries_mapping=region_countries_mapping,
        region_other_mapping=region_other_mapping
    )

    cbcr_foreign_scaled = scale_wxd_values(
        foreign_data=cbcr_foreign_scaled2,
        variables=scaling_variables,
        wxd_jurisdictions=wxd_jurisdictions,
        region_countries_mapping=region_countries_mapping
    )

    # Step 5e: Combine all scaled variables into one
    for variable in scaling_variables:
        scaled_continent_var = f'scaled_continent_fitted_{variable}'
        scaled_other_var = f'scaled_other_fitted_{variable}'
        scaled_wxd_var = f'scaled_wxd_fitted_{variable}'

        cbcr_foreign_scaled[f'scaled_fitted_{variable}'] = (
            cbcr_foreign_scaled[scaled_continent_var] +
            cbcr_foreign_scaled[scaled_other_var] +
            cbcr_foreign_scaled[scaled_wxd_var]
        )

    # Step 6: Combine Domestic and Foreign Data
    cbcr_with_imputed_values = pd.concat([cbcr_domestic_imputed, cbcr_foreign_scaled], ignore_index=True)

    # Step 7: Substitute missing with imputed and/or scaled variables
    for variable in scaling_variables:
        actual_var = variable
        scaled_var = f'scaled_fitted_{variable}'
        fitted_var = f'fitted_{variable}'

        actual_missing_mask = cbcr_with_imputed_values[actual_var].isna()
        scaled_not_missing_mask = cbcr_with_imputed_values[scaled_var].notna()
        mask_replace_with_scaled = actual_missing_mask & scaled_not_missing_mask
        cbcr_with_imputed_values.loc[mask_replace_with_scaled, actual_var] = cbcr_with_imputed_values.loc[mask_replace_with_scaled, scaled_var]

        actual_missing_mask = cbcr_with_imputed_values[actual_var].isna()  # Update mask after previous substitution
        cbcr_with_imputed_values.loc[actual_missing_mask, actual_var] = cbcr_with_imputed_values.loc[actual_missing_mask, fitted_var]

    # Step 8: Check aggregate values after imputation and scaling
    for (iso_parent, year), group in cbcr_with_imputed_values.groupby(['iso_parent', 'year']):
        if (group['original_cbcr_data'] == 1).any():
            original_aggregate = cbcr_with_imputed_values[
                (cbcr_with_imputed_values['iso_parent'] == iso_parent) & 
                (cbcr_with_imputed_values['year'] == year) & 
                (cbcr_with_imputed_values['original_cbcr_data'] == 1)
            ][scaling_variables].sum()

            scaled_aggregate = group[~group['iso_partner'].isin(non_countries)][scaling_variables].sum()

            if not original_aggregate.equals(scaled_aggregate):
                print(f"Discrepancy found for iso_parent '{iso_parent}' in year '{year}':")
                print(f"  Original Aggregate: {original_aggregate}")
                print(f"  Scaled Aggregate: {scaled_aggregate}")

    # Step 9: Keep only required columns and rows for final DataFrame
    cbcr_with_imputed_values = cbcr_with_imputed_values[~cbcr_with_imputed_values['iso_partner'].isin(non_countries)]
    columns_to_keep = ['iso_parent', 'iso_partner', 'year', 'original_cbcr_data'] + scaling_variables
    cbcr_with_imputed_values = cbcr_with_imputed_values[columns_to_keep]

    # Step 10: Create a filtered version of cbcr_sample without target variables and iso_parent
    columns_to_drop = [
        'iso_parent', 'parent_jurisdiction', 'n_cbcr', 'n_cbcr_groups', 'n_entities',
        'profit_loss_before_income_tax', 'ln_profit_loss_before_income_tax',
        'profit_loss_before_income_tax_corrected', 'ln_profit_loss_before_income_tax_corrected',
        'adjusted_profit_loss_before_income_tax', 'ln_adjusted_profit_loss_before_income_tax',
        'income_tax_paid_on_cash_basis', 'ln_income_tax_paid_on_cash_basis',
        'income_tax_accrued_current_year'
    ] + target_variables + scaling_variables
    cbcr_sample_filtered = cbcr_sample.drop(columns=columns_to_drop).drop_duplicates()
    cbcr_sample_filtered = cbcr_sample_filtered[~cbcr_sample_filtered['iso_partner'].isin(non_countries)]

    # Step 11: Merge additional variables from cbcr_sample based on iso_partner and year
    cbcr_with_imputed_values = cbcr_with_imputed_values.merge(
        cbcr_sample_filtered,
        on=['iso_partner', 'year'],
        how='left'
    )

    # Step 12: Create payroll variable
    cbcr_with_imputed_values['payroll'] = cbcr_with_imputed_values['n_employees'] * cbcr_with_imputed_values['wage_monthly'] * 12 

    # Append the yearly results to the final results list
    final_results.append(cbcr_with_imputed_values)

# Combine all yearly results into a single DataFrame
final_dataset = pd.concat(final_results, ignore_index=True)

# Save the final dataset to a CSV or any other format
final_dataset.to_csv(f'{data_final}/cbcr_with_imputed_values.csv', index=False)

print("Imputation and scaling process completed for all years.")


In [ ]:
# Check duplicates
def check_duplicates(df, name):
    if df.duplicated().any():
        print(f"Warning: {name} contains duplicate rows.")
    else:
        print(f"No duplicates found in {name}.")

check_duplicates(cbcr_with_imputed_values[['iso_parent', 'iso_partner','year']], "dataset with imputed values")

In [144]:
cbcr_with_imputed_values.to_csv(f'{data_final}/cbcr_with_imputed_values.csv')

#### 4.2 Generate and save bootstrapped samples with imputed values

In [145]:
def generate_sample_with_imputed_values(cbcr_imputation_sample, non_countries, scaling_variables, 
                   region_countries_mapping, region_other_mapping, wxd_jurisdictions, 
                   target_variables, first_year, n_years):
    """Process bootstrap samples with imputed values for multiple years."""
    
    # Initialize a list to collect results for each year
    final_results = []

    # Iterate over the range of years
    for year in range(first_year, first_year + n_years):
        print(f"Processing year: {year}")

        # Step 1: Split the DataFrame for the current year
        cbcr_yearly_data = cbcr_imputation_sample[cbcr_imputation_sample['year'] == year]
        
        domestic_mask = cbcr_yearly_data['iso_parent'] == cbcr_yearly_data['iso_partner']
        foreign_mask = cbcr_yearly_data['iso_parent'] != cbcr_yearly_data['iso_partner']

        cbcr_domestic = cbcr_yearly_data[domestic_mask].copy()
        cbcr_foreign = cbcr_yearly_data[foreign_mask].copy()

        # Step 2: Drop rows which we do not need to impute
        #partner_year_groups = cbcr_foreign.groupby(['iso_partner', 'year'])
        #for (partner, year), group in partner_year_groups:
        #    if (group['original_cbcr_data'] == 1).any() and not any(group['iso_partner'].isin(non_countries)):
        #        cbcr_foreign = cbcr_foreign[~((cbcr_foreign['iso_partner'] == partner) & 
        #                                       (cbcr_foreign['year'] == year) & 
        #                                       (cbcr_foreign['original_cbcr_data'] == 0))]

        # Step 3: Impute Domestic Values
        cbcr_domestic_imputed = predict_domestic_values(cbcr_domestic)

        # Step 4: Impute Foreign Values
        cbcr_foreign_imputed = predict_foreign_values(cbcr_foreign)

        # Step 5: Scaling for different groups
        cbcr_foreign_scaled1 = scale_predicted_values_continents(
            filled_foreign_data=cbcr_foreign_imputed,
            variables=scaling_variables,
            region_countries_mapping=region_countries_mapping
        )

        cbcr_foreign_scaled2 = scale_other_predicted_values(
            filled_foreign_data=cbcr_foreign_scaled1,
            variables=scaling_variables,
            region_countries_mapping=region_countries_mapping,
            region_other_mapping=region_other_mapping
        )

        cbcr_foreign_scaled = scale_wxd_values(
            foreign_data=cbcr_foreign_scaled2,
            variables=scaling_variables,
            wxd_jurisdictions=wxd_jurisdictions,
            region_countries_mapping=region_countries_mapping
        )

        # Step 5e: Combine all scaled variables into one
        for variable in scaling_variables:
            scaled_continent_var = f'scaled_continent_fitted_{variable}'
            scaled_other_var = f'scaled_other_fitted_{variable}'
            scaled_wxd_var = f'scaled_wxd_fitted_{variable}'
            
            cbcr_foreign_scaled[f'scaled_fitted_{variable}'] = (
                cbcr_foreign_scaled[scaled_continent_var] +
                cbcr_foreign_scaled[scaled_other_var] +
                cbcr_foreign_scaled[scaled_wxd_var]
            )

        # Step 6: Combine Domestic and Foreign Data
        cbcr_with_imputed_values = pd.concat([cbcr_domestic_imputed, cbcr_foreign_scaled], ignore_index=True)

        # Step 7: Substitute missing with imputed and/or scaled variables
        for variable in scaling_variables:
            actual_var = variable
            scaled_var = f'scaled_fitted_{variable}'
            fitted_var = f'fitted_{variable}'

            actual_missing_mask = cbcr_with_imputed_values[actual_var].isna()
            scaled_not_missing_mask = cbcr_with_imputed_values[scaled_var].notna()
            mask_replace_with_scaled = actual_missing_mask & scaled_not_missing_mask
            cbcr_with_imputed_values.loc[mask_replace_with_scaled, actual_var] = cbcr_with_imputed_values.loc[mask_replace_with_scaled, scaled_var]

            actual_missing_mask = cbcr_with_imputed_values[actual_var].isna()  # Update mask after previous substitution
            cbcr_with_imputed_values.loc[actual_missing_mask, actual_var] = cbcr_with_imputed_values.loc[actual_missing_mask, fitted_var]

        # Step 8: Check aggregate values after imputation and scaling
        for (iso_parent, year), group in cbcr_with_imputed_values.groupby(['iso_parent', 'year']):
            if (group['original_cbcr_data'] == 1).any():
                original_aggregate = cbcr_with_imputed_values[
                    (cbcr_with_imputed_values['iso_parent'] == iso_parent) & 
                    (cbcr_with_imputed_values['year'] == year) & 
                    (cbcr_with_imputed_values['original_cbcr_data'] == 1)
                ][scaling_variables].sum()

                scaled_aggregate = group[~group['iso_partner'].isin(non_countries)][scaling_variables].sum()

                if not original_aggregate.equals(scaled_aggregate):
                    print(f"Discrepancy found for iso_parent '{iso_parent}' in year '{year}':")
                    print(f"  Original Aggregate: {original_aggregate}")
                    print(f"  Scaled Aggregate: {scaled_aggregate}")

        # Step 9: Keep only required columns and rows for final DataFrame
        cbcr_with_imputed_values = cbcr_with_imputed_values[~cbcr_with_imputed_values['iso_partner'].isin(non_countries)]
        columns_to_keep = ['iso_parent', 'iso_partner', 'year', 'original_cbcr_data'] + scaling_variables
        cbcr_with_imputed_values = cbcr_with_imputed_values[columns_to_keep]

        # Step 10: Create a filtered version of cbcr_sample without target variables and iso_parent
        columns_to_drop = [
            'iso_parent', 'parent_jurisdiction', 'n_cbcr', 'n_cbcr_groups', 'n_entities',
            'profit_loss_before_income_tax', 'ln_profit_loss_before_income_tax',
            'profit_loss_before_income_tax_corrected', 'ln_profit_loss_before_income_tax_corrected',
            'adjusted_profit_loss_before_income_tax', 'ln_adjusted_profit_loss_before_income_tax',
            'income_tax_paid_on_cash_basis', 'ln_income_tax_paid_on_cash_basis',
            'income_tax_accrued_current_year'
        ] + target_variables + scaling_variables
        cbcr_sample_filtered = cbcr_sample.drop(columns=columns_to_drop).drop_duplicates()
        cbcr_sample_filtered = cbcr_sample_filtered[~cbcr_sample_filtered['iso_partner'].isin(non_countries)]

        # Step 11: Merge additional variables from cbcr_sample based on iso_partner and year
        cbcr_with_imputed_values = cbcr_with_imputed_values.merge(
            cbcr_sample_filtered,
            on=['iso_partner', 'year'],
            how='left'
        )

        # Step 12: Create payroll variable
        cbcr_with_imputed_values['payroll'] = cbcr_with_imputed_values['n_employees'] * cbcr_with_imputed_values['wage_monthly'] * 12

        # Append the yearly results to the final results list
        final_results.append(cbcr_with_imputed_values)

    # Combine all yearly results into a single DataFrame
    final_dataset = pd.concat(final_results, ignore_index=True)

    return final_dataset  # Return the processed DataFrame

In [146]:
# Main function to generate multiple samples
def generate_multiple_samples(cbcr_imputation_sample, non_countries, scaling_variables, 
                              region_countries_mapping, region_other_mapping, 
                              wxd_jurisdictions, target_variables, n_samples=1000):
    all_imputed_samples = []  # List to hold all processed samples

    for sample_number in range(1, n_samples + 1):
        print(f"Generating sample {sample_number}/{n_samples}")

        # Use the original cbcr_imputation_sample without resampling
        imputed_values = generate_sample_with_imputed_values(
            cbcr_imputation_sample = cbcr_imputation_sample, 
            non_countries = non_countries, 
            scaling_variables = scaling_variables, 
            region_countries_mapping = region_countries_mapping, 
            region_other_mapping = region_other_mapping, 
            wxd_jurisdictions = wxd_jurisdictions, 
            target_variables = target_variables,
            first_year = first_year,
            n_years = n_years
        )
        
        # Add a sample identifier
        imputed_values['sample_id'] = sample_number

        # Append the processed sample to the list
        all_imputed_samples.append(imputed_values)

    # Combine all samples into one DataFrame
    combined_imputed_samples = pd.concat(all_imputed_samples, ignore_index=True)

    # Save to a single CSV file
    combined_imputed_samples.to_csv(f'{data_final}/cbcr_with_imputed_values_multiple_samples.csv', index=False)

    return combined_imputed_samples  # Return the combined DataFrame

In [ ]:
# Call the function
imputed_samples = generate_multiple_samples(
    cbcr_imputation_sample,
    non_countries,
    scaling_variables,
    region_countries_mapping,
    region_other_mapping,
    wxd_jurisdictions,
    target_variables,
    n_samples=3  # Set the desired number of samples
)
